# Summertime Map

Every recording here is the same song, so the differences the map picks up come from
interpretation: tempo, energy, instrumentation, mood.

Steps:
1. Load the song list
2. Find each recording on Deezer (album cover + 30-second preview)
3. Audio features from each preview
4. Standardize
5. PCA: the 2-D map
6. K-Means: are there natural groups?
7. How valid are the groups?
8. What might each group represent?
9. Export for the website

In [1]:
from pathlib import Path
import re
import time
import unicodedata

import httpx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path("../data")
CLIPS = DATA / "clips"          # downloaded previews (git-ignored)
CLIPS.mkdir(exist_ok=True)
FEATURES = [
    "danceability", "energy", "valence", "tempo", "loudness",
    "acousticness", "instrumentalness", "speechiness", "liveness",
]
http = httpx.Client(timeout=30)

## 1. Load the song list

`data/songs.csv` has one row per recording. The Spotify track ID is taken from the link.

In [2]:
songs = pd.read_csv(DATA / "songs.csv")

def spotify_id(url: str) -> str:
    m = re.search(r"track[/:]([A-Za-z0-9]{22})", str(url))
    if not m:
        raise ValueError(f"Not a Spotify track link: {url}")
    return m.group(1)

songs["track_id"] = songs["spotify_url"].map(spotify_id)
print(f"{len(songs)} recordings")
songs

33 recordings


,artist,title,year,year_source,duration_s,spotify_url,notes,track_id
0,Big Brother & The Holding Company;Janis Joplin,Summertime,1968.0,draft - verify,241,https://open.spotify.com/track/3eYCJXTViFwl5md...,NaN,3eYCJXTViFwl5mdtcTsF4G
1,Ella Fitzgerald;Louis Armstrong,Summertime,1957.0,draft - verify,298,https://open.spotify.com/track/2gNjmvuQiEd2z9S...,NaN,2gNjmvuQiEd2z9SqyYi8HH
2,The Zombies,Summertime,1965.0,draft - verify,138,https://open.spotify.com/track/7egTKPYxbWaaV1j...,NaN,7egTKPYxbWaaV1jEzb1rHu
3,Sam Cooke,Summertime,1957.0,draft - verify,143,https://open.spotify.com/track/5blqPmozIU2vhHN...,NaN,5blqPmozIU2vhHN7vbBKxk
4,Sublime,Doin' Time,1996.0,draft - verify,251,https://open.spotify.com/track/4mAqG6KNWZQiekS...,based on Summertime,4mAqG6KNWZQiekSthr2QtL
5,Charlie Parker,Summertime,1949.0,draft - verify,167,https://open.spotify.com/track/6cU4urQRUieTBH0...,NaN,6cU4urQRUieTBH08yKHAqj
6,Al Green,Summertime,1969.0,draft - verify,179,https://open.spotify.com/track/1wgyVw5XL14SW2B...,NaN,1wgyVw5XL14SW2BV4lBcVi
7,Lana Del Rey,Doin' Time,2019.0,draft - verify,202,https://open.spotify.com/track/0cLS7FslBJgsuWw...,cover of Sublime's Doin' Time,0cLS7FslBJgsuWwJNRnPWB
8,Doc Watson;David Grisman,Summertime,1997.0,draft - verify,260,https://open.spotify.com/track/5xV7bCTvuIcxvB1...,NaN,5xV7bCTvuIcxvB1NcjshNY
9,Billy Stewart,Summertime,1966.0,draft - verify,255,https://open.spotify.com/track/4qH8fQ5Ub2r6llH...,NaN,4qH8fQ5Ub2r6llHnrRGM3w


## 2. Find each recording on Deezer (or iTunes)

Spotify no longer gives new apps preview clips, but Deezer does, along with album covers.
Many songs exist in several versions (remasters, live takes, instrumentals), so for each
song we search Deezer and keep the candidate that:

1. is by the same artist and has the same title,
2. isn't tagged *instrumental / remix / live / dub* unless the Spotify title is too, and
3. has the length closest to the Spotify track's.

Matches are saved in `data/matches.csv` and reused on later runs, so you can fix a wrong one by
editing its `source` and `source_id` there (the number at the end of the track's deezer.com
or music.apple.com link).

**Check the table below by eye**, starting with the biggest length differences: a wrong
match means the wrong recording gets analyzed.

If Deezer doesn't have a recording, the same search runs on Apple's iTunes Search API,
which also has 30-second previews. Checked on Sam Cooke's version, which both have:
the features from the two previews differ only slightly (e.g. acousticness 0.74 vs
0.85, tempo 118 vs 115), far less than the songs differ from each other.

In [3]:
TAGS = ["instrumental", "remix", "live", "dub", "karaoke", "demo"]
MATCHES = DATA / "matches.csv"

def norm(s: str) -> str:
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()

def base_title(title: str) -> str:
    """'Summertime - Mono Version' / 'Summertime (Live)' / 'Summertime / Motherless Child' -> 'summertime'"""
    return norm(re.split(r" - |\(|\[|/", title)[0]).strip()

def deezer_match(artist: str, title: str, duration_s: int) -> dict | None:
    main = artist.split(";")[0]
    short = base_title(title)
    cands = []
    for q in [f'artist:"{main}" track:"{short}"', f"{main} {short}"]:
        cands += http.get("https://api.deezer.com/search", params={"q": q, "limit": 25}).json().get("data", [])
    same_artist = lambda c: norm(main) in norm(c["artist"]["name"]) or norm(c["artist"]["name"]) in norm(artist)
    same_title = lambda c: short in norm(c["title"])
    extra_tag = lambda c: any(t in norm(c["title"]) and t not in norm(title) for t in TAGS)
    cands = [c for c in cands if same_artist(c) and same_title(c)]
    if not cands:
        return None
    return min(cands, key=lambda c: (extra_tag(c), abs(c["duration"] - duration_s)))

def itunes_match(artist: str, title: str, duration_s: int) -> dict | None:
    main = artist.split(";")[0]
    short = base_title(title)
    r = http.get("https://itunes.apple.com/search",
                 params={"term": f"{main} {short}", "media": "music", "entity": "song", "limit": 25})
    cands = [c for c in r.json().get("results", [])
             if (norm(main) in norm(c["artistName"]) or norm(c["artistName"]) in norm(artist))
             and short in norm(c["trackName"]) and c.get("previewUrl")]
    extra_tag = lambda c: any(t in norm(c["trackName"]) and t not in norm(title) for t in TAGS)
    if not cands:
        return None
    return min(cands, key=lambda c: (extra_tag(c), abs(c["trackTimeMillis"] / 1000 - duration_s)))

def find_recording(s) -> dict:
    if m := deezer_match(s.artist, s.title, s.duration_s):
        return {"source": "deezer", "source_id": m["id"],
                "match_title": f'{m["artist"]["name"]} - {m["title"]}', "album": m["album"]["title"],
                "cover": m["album"]["cover_big"], "length_diff_s": m["duration"] - s.duration_s}
    if m := itunes_match(s.artist, s.title, s.duration_s):
        return {"source": "itunes", "source_id": m["trackId"],
                "match_title": f'{m["artistName"]} - {m["trackName"]}', "album": m["collectionName"],
                "cover": m["artworkUrl100"].replace("100x100", "500x500"),
                "length_diff_s": round(m["trackTimeMillis"] / 1000) - s.duration_s}
    return {}

# Keep earlier matches (so a match you fixed by hand in matches.csv stays fixed);
# only look up songs that haven't been matched yet.
COLS = ["track_id", "source", "source_id", "match_title", "album", "cover", "length_diff_s"]
matches = pd.read_csv(MATCHES) if MATCHES.exists() else pd.DataFrame(columns=COLS)
matches = matches.dropna(subset=["source_id"])
new = []
for s in songs[~songs["track_id"].isin(matches["track_id"])].itertuples():
    new.append({"track_id": s.track_id, **find_recording(s)})
if new:
    matches = pd.concat([matches, pd.DataFrame(new)], ignore_index=True)
    matches.to_csv(MATCHES, index=False)
print(f"{len(new)} newly matched")

view = songs[["track_id", "artist", "title"]].merge(matches, on="track_id", how="left")
unmatched = view[view["source_id"].isna()]
if len(unmatched):
    print("No match on Deezer or iTunes for:", "; ".join(unmatched["artist"] + " - " + unmatched["title"]))
# Suspicious matches first: biggest length differences.
view.drop(columns=["track_id", "cover"]).sort_values("length_diff_s", key=abs, ascending=False)

0 newly matched


,artist,title,source,source_id,match_title,album,length_diff_s
18,Nina Simone,Summertime,deezer,1.137982e+09,Nina Simone - Summertime,Soft Silk - My Baby Just Cares For Me,23.0
15,The Ventures,Summertime,deezer,1.526610e+09,The Ventures - Summertime,Beach Party,-6.0
30,Sidney Bechet,Summertime,deezer,4.202661e+09,Sidney Bechet - Summertime (2026 Remaster),Classics in Swingtime (2026 Remaster),5.0
0,Big Brother & The Holding Company;Janis Joplin,Summertime,deezer,1.561315e+07,Big Brother & The Holding Company - Summertime,The Essential Janis Joplin,-3.0
11,Mahalia Jackson,Summertime / Sometimes I Feel Like a Motherles...,deezer,1.062493e+08,Mahalia Jackson - Summertime / Sometimes I Fee...,Bless This House,-3.0
10,Rosinha De Valença,Summertime,deezer,9.193236e+07,Rosinha De Valença - Summertime,Um Violão em Primeiro Plano,2.0
16,BB Seaton,Summertime,itunes,5.196697e+08,BB Seaton - Summertime,Summertime (Save It For a Rainy Day) - Single,2.0
3,Sam Cooke,Summertime,deezer,1.199407e+08,Sam Cooke - Summertime,The Best of Sam Cooke,-2.0
13,Miles Davis,Summertime - Mono Version,deezer,8.723560e+05,Miles Davis - Summertime,Porgy And Bess,-2.0
19,The New Stan Getz Quartet;George Gershwin,Summertime,deezer,3.255112e+09,The New Stan Getz Quartet - Summertime (Live A...,"Getz Au Go Go (Live At Café Au Go Go, 1964 / 2...",2.0


## 3. Audio features from each preview

ReccoBeats' full-track features only cover 6 of these 11 recordings, and features
computed from a 30-second clip don't agree with full-track ones (tempo and
acousticness especially), so mixing the two would distort the map. Instead, **every**
song is analyzed the same way: its preview is uploaded to ReccoBeats'
audio-analysis endpoint.

So the map describes the 30-second excerpt, which is also what the website plays.
The catch: if a preview happens to be an intro or a solo, it may not represent the
whole performance.

Results are cached in `data/features.csv`; delete a row to re-analyze that song.

In [4]:
ANALYZE_URL = "https://api.reccobeats.com/v1/analysis/audio-features"
CACHE = DATA / "features.csv"

def preview_url(source: str, source_id: int) -> str:
    # Preview links expire, so ask for a fresh one each time.
    if source == "deezer":
        return http.get(f"https://api.deezer.com/track/{source_id}").json()["preview"]
    return http.get("https://itunes.apple.com/lookup", params={"id": source_id}).json()["results"][0]["previewUrl"]

def analyze_clip(source: str, source_id: int) -> dict:
    ext, mime = ("mp3", "audio/mpeg") if source == "deezer" else ("m4a", "audio/mp4")
    clip = CLIPS / f"{source}-{source_id}.{ext}"
    if not clip.exists():
        clip.write_bytes(http.get(preview_url(source, source_id)).content)
    for attempt in range(3):
        with clip.open("rb") as f:
            r = http.post(ANALYZE_URL, files={"audioFile": (clip.name, f, mime)})
        if r.status_code == 429:  # rate limited: wait and retry
            time.sleep(min(int(r.headers.get("Retry-After", "5")), 30))
            continue
        r.raise_for_status()
        return {f: r.json()[f] for f in FEATURES}
    raise RuntimeError(f"ReccoBeats kept rate-limiting {source} {source_id}")

cached = pd.read_csv(CACHE) if CACHE.exists() else pd.DataFrame(columns=["track_id", *FEATURES])
for m in matches.dropna(subset=["source_id"]).itertuples():
    if m.track_id not in set(cached["track_id"]):
        row = {"track_id": m.track_id, **analyze_clip(m.source, int(m.source_id))}
        cached = pd.concat([cached, pd.DataFrame([row])], ignore_index=True)
        cached.to_csv(CACHE, index=False)

cached[FEATURES] = cached[FEATURES].astype(float)
df = songs.merge(matches, on="track_id").merge(cached, on="track_id", how="left")
missing = df[df[FEATURES].isna().any(axis=1)]
if len(missing):
    print("No features for:", ", ".join(missing["artist"]))
df = df.dropna(subset=FEATURES).reset_index(drop=True)
print(f"{len(df)} recordings with features")
df[["artist", "title", *FEATURES]].round(2)

33 recordings with features


,artist,title,danceability,energy,valence,tempo,loudness,acousticness,instrumentalness,speechiness,liveness
0,Big Brother & The Holding Company;Janis Joplin,Summertime,0.51,0.40,0.45,101.33,-9.89,0.26,0.01,0.04,0.14
1,Ella Fitzgerald;Louis Armstrong,Summertime,0.30,0.09,0.14,111.75,-15.89,0.97,0.88,0.04,0.10
2,The Zombies,Summertime,0.49,0.20,0.20,107.24,-15.16,0.84,0.02,0.04,0.11
3,Sam Cooke,Summertime,0.60,0.33,0.45,115.09,-11.82,0.85,0.04,0.04,0.14
4,Sublime,Doin' Time,0.63,0.80,0.67,71.68,-3.66,0.03,0.00,0.11,0.17
5,Charlie Parker,Summertime,0.43,0.23,0.48,140.65,-10.35,0.97,0.06,0.04,0.13
6,Al Green,Summertime,0.62,0.29,0.53,137.08,-12.75,0.79,0.00,0.05,0.13
7,Lana Del Rey,Doin' Time,0.65,0.53,0.48,145.87,-9.27,0.48,0.01,0.04,0.13
8,Doc Watson;David Grisman,Summertime,0.57,0.16,0.39,131.02,-14.57,0.94,0.02,0.04,0.12
9,Billy Stewart,Summertime,0.71,0.66,0.81,111.59,-8.10,0.42,0.00,0.06,0.14


## 4. Standardize

Features are on very different scales (tempo in BPM, loudness in dB, the rest 0–1). Z-score each one so no feature dominates just because of its units.

In [5]:
# TODO

## 5. PCA: the 2-D map

Project the nine standardized features down to two axes, plot each recording as a labelled point, and draw arrows showing which features drive each axis.

In [6]:
# TODO

## 6. K-Means: are there natural groups?

Cluster on all nine standardized features (not on the 2-D map), trying a few values of k and comparing silhouette scores.

In [7]:
# TODO

## 7. How valid are the groups?

Silhouette score per cluster, plus a stability test: re-cluster random subsets of the songs and measure how often the same recordings end up together.

In [8]:
# TODO

## 8. What might each group represent?

Compare each cluster's average features with the whole set and draft a plain-language label from the biggest differences. Edit these by ear.

In [9]:
# TODO

## 9. Export for the website

Write songs, coordinates, clusters, album covers and preview links to `site/data.json`.

In [10]:
# TODO